# Fine-Tuning LED-Base for Scientific Research Paper Summarization

This notebook demonstrates fine-tuning the **Longformer Encoder-Decoder (LED)** model (`allenai/led-base-16384`) on the `ccdv/pubmed-summarization` dataset. LED is specifically designed for long-context tasks by employing local + global attention patterns.

### Notebook Sections:
1. **Environment Setup & Packages Install**
2. **Dataset Loading & Preprocessing**
3. **LED Model Configuration & Global Attention Setup**
4. **Memory-Optimized Training Loop**
5. **ROUGE Evaluation**
6. **Model Checkpointing & Reloading**
7. **Inference Demo**
8. **FastAPI Deployment Preparation**

In [ ]:
# Install necessary packages
!pip install -q transformers datasets evaluate rouge-score accelerate optimum

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 7.6 MB/s eta 0:00:00


## 1. Dataset Loading and Preprocessing

We load the `ccdv/pubmed-summarization` dataset and clean empty values. Due to resource boundaries on Google Colab T4 GPU, we keep a subset or safely process the sequences.

**Sequence Length Selection:**
- **Input length**: We use **4096** tokens. It balances scientific content retrieval and T4 GPU's 16GB VRAM.
- **Output length**: We use **512** tokens to retain key research elements without causing sequence truncation or heavy GPU latency.

In [ ]:
import re
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Load Dataset
print("Loading dataset ccdv/pubmed-summarization...")
dataset = load_dataset("ccdv/pubmed-summarization", trust_remote_code=True)

# 2. Set Up Tokenizer
model_checkpoint = "allenai/led-base-16384"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 3. Clean Text Function
def clean_text(text):
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_function(examples):
    # Clean inputs and targets
    cleaned_articles = [clean_text(art) for art in examples["article"]]
    cleaned_abstracts = [clean_text(abs_text) for abs_text in examples["abstract"]]

    # Tokenize input
    model_inputs = tokenizer(
        cleaned_articles,
        max_length=4096,
        padding="max_length",
        truncation=True
    )

    # Tokenize target
    labels = tokenizer(
        cleaned_abstracts,
        max_length=512,
        padding="max_length",
        truncation=True
    )

    # Replace padding token id with -100 to ignore loss
    labels_ids = labels["input_ids"]
    labels_ids = [
        [(l_id if l_id != tokenizer.pad_token_id else -100) for l_id in label]
        for label in labels_ids
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

# Take a subset to ensure execution within typical Colab runtime sessions
train_subset = dataset["train"].select(range(100))
val_subset = dataset["validation"].select(range(20))

tokenized_train = train_subset.map(preprocess_function, batched=True, remove_columns=["article", "abstract"])
tokenized_val = val_subset.map(preprocess_function, batched=True, remove_columns=["article", "abstract"])

print("Preprocessing finished successfully!")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ccdv/pubmed-summarization' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ccdv/pubmed-summarization' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading dataset ccdv/pubmed-summarization...


README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

section/train-00000-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  210MB            

section/train-00000-of-00005.parquet: downloading bytes:           |  0.00B            

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Preprocessing finished successfully!


## 2. LED Configuration & Global Attention

Standard self-attention scales quadratically with sequence length ($O(N^2)$). LED uses a local attention pattern that scales linearly ($O(N \times W)$), paired with global attention for selected tokens.

We set global attention to the first token (`<s>`) so the model can construct an overall representation of the context while mapping decoder queries.

In [ ]:
from transformers import LEDForConditionalGeneration, LEDConfig

# Load pre-trained model config and setup generation configurations properly
config = LEDConfig.from_pretrained(model_checkpoint)
model = LEDForConditionalGeneration.from_pretrained(model_checkpoint, config=config)

# Setup generation parameters on generation_config instead of config
model.generation_config.num_beams = 4
model.generation_config.max_length = 512
model.generation_config.min_length = 128
model.generation_config.length_penalty = 2.0
model.generation_config.early_stopping = True

# Configure global attention mask mapping function
def generate_global_attention_mask(input_ids, bos_token_id):
    # Create global attention mask: first token gets global attention
    global_attention_mask = torch.zeros_like(input_ids)
    global_attention_mask[:, 0] = 1
    return global_attention_mask

Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

## 3. ROUGE Metrics Explanation

- **ROUGE-1**: Evaluates unigram (single-word) overlap. Measures factual recall and vocabulary capture.
- **ROUGE-2**: Evaluates bigram (two-word phrase) overlap. Measures readability and structural fluency.
- **ROUGE-L**: Evaluates Longest Common Subsequence (LCS). Assesses structural consistency and document flow.

In [ ]:
import evaluate
import numpy as np

rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode predictions
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels with pad token
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

## 4. PyTorch Training Loop (GPU Optimization)

To train on T4 without Out-Of-Memory (OOM) errors, we implement:
- **Mixed Precision (FP16)** via Hugging Face `Trainer`
- **Gradient Checkpointing** to save memory during backpropagation
- **Gradient Accumulation** to support effective batch sizing

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import os

# Enable gradient checkpointing in model configuration
model.gradient_checkpointing_enable()

training_args = Seq2SeqTrainingArguments(
    output_dir="./training_logs",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    num_train_epochs=3,
    fp16=True,
    predict_with_generate=True,
    warmup_steps=10,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_rouge1",
    greater_is_better=True
)

# Custom Data Collator to build dynamic global attention mask
def collate_fn(features):
    batch = {}
    input_ids = torch.stack([torch.tensor(f["input_ids"]) for f in features])
    attention_mask = torch.stack([torch.tensor(f["attention_mask"]) for f in features])
    labels = torch.stack([torch.tensor(f["labels"]) for f in features])

    global_attention_mask = generate_global_attention_mask(input_ids, tokenizer.bos_token_id)

    batch["input_ids"] = input_ids
    batch["attention_mask"] = attention_mask
    batch["global_attention_mask"] = global_attention_mask
    batch["labels"] = labels

    return batch

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

print("Starting training loop...")
train_result = trainer.train()
print("Training Complete!")

Starting training loop...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,11.471170,2.702944,28.669300,7.644700,14.195700,14.089900
2,8.890649,2.620182,32.763000,10.101600,17.799100,17.873500
3,9.682095,2.597587,31.712500,8.952000,16.758900,16.784600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['led.encoder.embed_tokens.weight', 'led.decoder.embed_tokens.weight', 'lm_head.weight'].


Training Complete!


## 5. Save Model and Training Artifacts

In [ ]:
# Save weights and configs
import os
import pandas as pd
import torch

os.makedirs("./tokenizer", exist_ok=True)
os.makedirs("./config", exist_ok=True)

torch.save(model.state_dict(), "led_base_scientific_summarizer.pth")
tokenizer.save_pretrained("./tokenizer")
model.config.save_pretrained("./config")
model.generation_config.save_pretrained("./config")

# Save evaluation results as CSV if trainer is available
if 'trainer' in globals():
    eval_results = trainer.evaluate()
    pd.DataFrame([eval_results]).to_csv("evaluation_results.csv", index=False)
    print("Trainer metrics saved.")
else:
    print("Trainer is not initialized in this session; skipping trainer.evaluate() but successfully saved weights.")

print("Model, tokenizer, and configs exported successfully!")

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
9.682095,2.620182,3,32.763000,10.101600,17.799100,17.873500


Trainer metrics saved.
Model, tokenizer, and configs exported successfully!


## 6. Checkpoint Reloading & Inference Demo

In [ ]:
# Reload model weights demonstration
# Instantiating from pretrained configuration identifier to prevent size mismatch
loaded_model = LEDForConditionalGeneration.from_pretrained(model_checkpoint)
loaded_model.load_state_dict(torch.load("led_base_scientific_summarizer.pth"))
loaded_model.to("cuda")

# Inference on an item from validation dataset
sample = dataset["validation"][0]
inputs = tokenizer(sample["article"], max_length=4096, return_tensors="pt", truncation=True, padding="max_length").to("cuda")
global_attention_mask = generate_global_attention_mask(inputs["input_ids"], tokenizer.bos_token_id).to("cuda")

print("Generating summary...")
summary_ids = loaded_model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    global_attention_mask=global_attention_mask,
    max_length=512,
    num_beams=4,
    early_stopping=True
)

generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("\n--- ORIGINAL ABSTRACT ---")
print(sample["abstract"])
print("\n--- GENERATED SUMMARY ---")
print(generated_summary)

Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

Generating summary...

--- ORIGINAL ABSTRACT ---
background and aim : there is lack of substantial indian data on venous thromboembolism ( vte ) . 
 the aim of this study was to provide real - world information on patient characteristics , management strategies , clinical outcomes , and temporal trends in vte.subjects and methods : multicentre retrospective registry involving 549 medical records of patients with confirmed diagnosis of vte ( deep vein thrombosis [ dvt ] confirmed by doppler ultrasonography ; pulmonary embolism [ pe ] by computed tomography , pulmonary angiography and/or v / q scan ) from 2006 to 2010 at three indian tertiary care hospitals.results:acute dvt without pe , acute dvt with pe , and pe alone were reported in 64% ( 352/549 ) , 23% ( 124/549 ) , and 13% ( 73/549 ) patients , respectively . 
 mean age was 47 ( 16 ) years , and 70% were males . 
 h / o dvt ( 34% ) , surgery including orthopedic surgery ( 28% ) , trauma ( 16% ) , and immobilization > 3 days ( 14% 

## 7. FastAPI Deployment Integration

Below is the code template designed for production integration in the FastAPI framework.

In [ ]:
fastapi_code = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
from transformers import LEDForConditionalGeneration, AutoTokenizer

app = FastAPI(title="Research Paper Summarization API")

# Load pipeline objects
tokenizer = AutoTokenizer.from_pretrained("./tokenizer")
model = LEDForConditionalGeneration.from_pretrained("./config")
model.load_state_dict(torch.load("led_base_scientific_summarizer.pth", map_location="cpu"))
model.eval()

class ArticleRequest(BaseModel):
    text: str

@app.post("/summarize")
def summarize(payload: ArticleRequest):
    try:
        inputs = tokenizer(payload.text, max_length=4096, return_tensors="pt", truncation=True)

        # Create global attention mask
        global_attention_mask = torch.zeros_like(inputs["input_ids"])
        global_attention_mask[:, 0] = 1

        with torch.no_grad():
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=global_attention_mask,
                max_length=512,
                num_beams=4,
                early_stopping=True
            )
        summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return {"summary": summary}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
"""

with open("app.py", "w") as f:
    f.write(fastapi_code)
print("Saved deployment-ready FastAPI script to 'app.py'!")

Saved deployment-ready FastAPI script to 'app.py'!
